### 01 - Instalação (Bibliotecas)

In [ ]:
%pip install numpy
%pip install matplotlib

### 02 - Importação (Recursos)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import csv

print(f"Numpy Version: {np.__version__}")
print(f"\nCSV Version: {csv.__version__}")

### 03 - Carregamento de Dados de Datasets

In [ ]:
def load_csv_data(file_path, columns_quantity):
  X, y = [], []

  with open(file_path, "r") as file:
    reader = csv.reader(file)

    next(reader) # Pulando a linha do cabeçalho.

    for row in reader:
      X_values = []

      for column in range(columns_quantity - 1):
        X_values.append(float(row[column]))

      X.append(X_values)

      y.append(int(row[columns_quantity - 1]))

    print(f"X: {X}")
    print(f"\nY: {y}")

  return np.array(X), np.array(y)

### 04 - Classe Auxiliar

In [ ]:
class Perceptron:
  def __init__(self, learning_rate = 0.01, epochs_quantity = 1000, weight_init = "random"):
    self.learning_rate = learning_rate
    self.epochs_quantity = epochs_quantity
    self.weights = None
    self.weight_init = weight_init
    self.bias = None
    self.errors_per_epoch = []

  def initialize_weights(self, features_quantity):
    if (self.weight_init == "random"):
      self.weights = np.random.rand(features_quantity)
      self.bias = np.random.rand()
    elif (self.weight_init == "zeros"):
      self.weights = np.zeros(features_quantity)
      self.bias = 0
    elif (self.weight_init == "normal"):
      self.weights = np.random.randn(features_quantity) * 0.01
      self.bias = np.random.randn() * 0.01
    else:
      raise ValueError("Opção inválida para inicialização dos pesos.")
    
  def predict(self, X):
    return np.sign(np.dot(X, self.weights) + self.bias)
  
  def predict_summary(self, X, y, epochs, phase):
    y_predict = self.predict(X)

    accuracy = np.mean(y_predict == y) * 100

    std_dev = np.std(self.weights)

    print(f"\n- Fase ({phase}):")

    print(f"\nAcurácia no Conjunto da Fase ({phase}): {accuracy:.2f}%")
    print(f"\nDesvio Padrão dos Pesos: {std_dev:.5f}")

    if (epochs > 0):
      print(f"\nÉpocas: {epochs}")

    print(f"\nPesos Finais Aprendidos: {self.weights}")
    print(f"\nBias Final: {self.bias}")

  def predict_2(self, X, y_true=None):
    y_predict = np.sign(np.dot(X, self.weights) + self.bias)

    if (y_true is not None):
      self.predict_summary(X, y_true, 0, "Teste")

    return y_predict

  def print_training_summary(self, X, y, epochs):
    y_predict = self.predict(X)

    accuracy = np.mean(y_predict == y) * 100

    print("\n- Treinamento:")

    print(f"\nAcurácia no Conjunto de Treino: {accuracy:.2f}%")
    print(f"\nÉpocas: {epochs}")
    print(f"\nPesos Finais Aprendidos: {self.weights}")
    print(f"\nBias Final: {self.bias}")

  def fit(self, X, y):
    samples_quantity, features_quantity = X.shape

    self.initialize_weights(features_quantity)

    self.bias = 0

    for epoch in range(self.epochs_quantity):
      errors = 0

      for sample in range(samples_quantity):
        linear_output = np.dot(X[sample], self.weights) + self.bias

        y_predict = np.sign(linear_output)

        if (y_predict != y[sample]):
          self.weights += self.learning_rate * y[sample] * X[sample]

          self.bias += self.learning_rate * y[sample]

          errors += 1

      self.errors_per_epoch.append(errors / samples_quantity)

      if (errors == 0):
        break

    self.print_training_summary(X, y, epoch + 1)

  def plot_errors(self):
    plt.figure(figsize=(8,4))

    plt.plot(self.errors_per_epoch, marker="o", linestyle="-")

    plt.xlabel("Época")
    plt.ylabel("Taxa de Erro")

    plt.title("Evolução dos Erros ao Longo das Épocas")

    plt.grid()

### 05 - Gráficos

In [ ]:
def plot_decision_boundary(X, y, model, title="Fronteira de Decisão do Perceptron"):
  x_min, x_max = X[:, 0].min() - 1, X[:, 0].max() + 1

  y_min, y_max = X[:, 1].min() - 1, X[:, 1].max() + 1

  xx, yy = np.meshgrid(np.linspace(x_min, x_max, 100), np.linspace(y_min, y_max, 100))

  Z = model.predict(np.c_[xx.ravel(), yy.ravel()])

  Z = Z.reshape(xx.shape)

  plt.contourf(xx, yy, Z, alpha=0.3)

  plt.scatter(X[:, 0], X[:, 1], c = y, cmap="bwr", edgecolors="k")

  plt.xlabel("X1")
  plt.ylabel("X2")

  plt.title(title)

  plt.show()

def plot_comparison(X, y, model):
  x_min, x_max = X[:, 0].min() - 1, X[:, 0].max() + 1

  y_min, y_max = X[:, 1].min() - 1, X[:, 1].max() + 1

  xx, yy = np.meshgrid(np.linspace(x_min, x_max, 100), np.linspace(y_min, y_max, 100))

  Z = model.predict(np.c_[xx.ravel(), yy.ravel()])

  Z = Z.reshape(xx.shape)

  fig, axis = plt.subplots(1, 2, figsize=(12,5))

  axis[0].contourf(xx, yy, Z, alpha=0.3)

  axis[0].scatter(X[:, 0], X[:, 1], c = y, cmap="bwr", edgecolors="k")

  axis[0].set_xlabel("X1")
  axis[0].set_ylabel("X2")

  axis[0].set_title("Fronteira de Decisão do Perceptron")

  axis[1].scatter(X[:, 0], X[:, 1], c = y, cmap="bwr", edgecolors="k")

  axis[1].set_xlabel("X1")
  axis[1].set_ylabel("X2")

  axis[1].set_title("Distribuição dos Dados do Dataset")

  plt.tight_layout()

  plt.show()

### 06 - Primeiro Dataset (Teste)

In [ ]:
dataset_1_test_X, dataset_1_test_y = load_csv_data("./Data/test_dataset_01.csv", 3)

dataset_1_test_y = np.where(dataset_1_test_y == -1, -1, 1)

dataset_1_test_perceptron = Perceptron(learning_rate=0.1, epochs_quantity=100)

dataset_1_test_perceptron.fit(dataset_1_test_X, dataset_1_test_y)

dataset_1_test_y_predict = dataset_1_test_perceptron.predict_2(dataset_1_test_X, dataset_1_test_y)

dataset_1_test_perceptron.plot_errors()

plot_comparison(dataset_1_test_X, dataset_1_test_y, dataset_1_test_perceptron)

### 07 - Primeiro Dataset (Treinamento)

In [ ]:
dataset_1_train_X, dataset_1_train_y = load_csv_data("./Data/train_dataset_01.csv", 3)

dataset_1_train_y = np.where(dataset_1_train_y == -1, -1, 1)

dataset_1_train_perceptron = Perceptron(learning_rate=0.1, epochs_quantity=100)

dataset_1_train_perceptron.fit(dataset_1_train_X, dataset_1_train_y)

dataset_1_train_y_predict = dataset_1_train_perceptron.predict_2(dataset_1_train_X, dataset_1_train_y)

dataset_1_train_perceptron.plot_errors()

plot_comparison(dataset_1_train_X, dataset_1_train_y, dataset_1_train_perceptron)

### 08 - Segundo Dataset (Teste)

In [ ]:
dataset_2_test_X, dataset_2_test_y = load_csv_data("./Data/test_dataset_02.csv", 3)

dataset_2_test_y = np.where(dataset_2_test_y == -1, -1, 1)

dataset_2_test_perceptron = Perceptron(learning_rate=0.1, epochs_quantity=100)

dataset_2_test_perceptron.fit(dataset_2_test_X, dataset_2_test_y)

dataset_2_test_y_predict = dataset_2_test_perceptron.predict_2(dataset_2_test_X, dataset_2_test_y)

dataset_2_test_perceptron.plot_errors()

plot_comparison(dataset_2_test_X, dataset_2_test_y, dataset_2_test_perceptron)

### 09 - Segundo Dataset (Treinamento)

In [ ]:
dataset_2_train_X, dataset_2_train_y = load_csv_data("./Data/train_dataset_02.csv", 3)

dataset_2_train_y = np.where(dataset_2_train_y == -1, -1, 1)

dataset_2_train_perceptron = Perceptron(learning_rate=0.1, epochs_quantity=100)

dataset_2_train_perceptron.fit(dataset_2_train_X, dataset_2_train_y)

dataset_2_train_y_predict = dataset_2_train_perceptron.predict_2(dataset_2_train_X, dataset_2_train_y)

dataset_2_train_perceptron.plot_errors()

plot_comparison(dataset_2_train_X, dataset_2_train_y, dataset_2_train_perceptron)